# Brain Tumor MRI Classification — ConvNeXt-Tiny Fine-Tuning
**Runtime: GPU → T4**

## 1. Setup

In [ ]:
!pip install -q timm imagehash scikit-learn matplotlib seaborn

In [ ]:
import os
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm
import imagehash
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

## 2. Dataset
Upload `dataset.zip` to Colab or use Kaggle API. Adjust `DATA_ROOT` accordingly.

In [ ]:
# Option A: Upload zip and unzip
# from google.colab import files
# uploaded = files.upload()  # upload dataset.zip
# !unzip -q dataset.zip -d /content/dataset

# Option B: Kaggle API
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d muratkokludataset/brain-tumor-multimodal-image-ct-and-mri -p /content/dataset --unzip

DATA_ROOT = Path("/content/dataset/Dataset/Brain Tumor MRI images")
assert DATA_ROOT.exists(), f"Dataset not found at {DATA_ROOT}"

for cls in sorted(os.listdir(DATA_ROOT)):
    count = len(list((DATA_ROOT / cls).glob("*")))
    print(f"{cls}: {count} images")

## 3. Deduplication (pHash)

In [ ]:
full_dataset = datasets.ImageFolder(root=str(DATA_ROOT))
print(f"Total images before dedup: {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}")
print(f"Class-to-idx: {full_dataset.class_to_idx}")

seen_hashes = {}
unique_indices = []
duplicate_count = 0

for idx in range(len(full_dataset)):
    img_path = full_dataset.imgs[idx][0]
    try:
        img = Image.open(img_path)
        h = str(imagehash.phash(img, hash_size=16))
        if h not in seen_hashes:
            seen_hashes[h] = idx
            unique_indices.append(idx)
        else:
            duplicate_count += 1
    except Exception:
        duplicate_count += 1

print(f"Duplicates removed: {duplicate_count}")
print(f"Unique images: {len(unique_indices)}")

## 4. Stratified Split (80/10/10)

In [ ]:
unique_labels = [full_dataset.targets[i] for i in unique_indices]

train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    unique_indices, unique_labels, test_size=0.2, stratify=unique_labels, random_state=42
)
val_idx, test_idx, val_labels, test_labels = train_test_split(
    temp_idx, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
print(f"Train class dist: {Counter(train_labels)}")
print(f"Val class dist:   {Counter(val_labels)}")
print(f"Test class dist:  {Counter(test_labels)}")

## 5. Transforms & DataLoaders

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_ds = datasets.ImageFolder(root=str(DATA_ROOT), transform=train_transform)
eval_ds = datasets.ImageFolder(root=str(DATA_ROOT), transform=eval_transform)

train_subset = Subset(train_ds, train_idx)
val_subset = Subset(eval_ds, val_idx)
test_subset = Subset(eval_ds, test_idx)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 6. Model — ConvNeXt-Tiny (Pretrained ImageNet)

In [ ]:
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=2)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

## 7. Training Utilities

In [ ]:
class_counts = Counter(train_labels)
total_train = len(train_labels)
num_classes = len(class_counts)
class_weights = torch.tensor(
    [total_train / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
).to(device)
print(f"Class weights: {class_weights}")

criterion = nn.CrossEntropyLoss(weight=class_weights)


def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    if scheduler:
        scheduler.step()
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

## 8. Stage 1 — Train Head Only (Backbone Frozen)

In [ ]:
for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 1 trainable params: {trainable:,}")

optimizer_s1 = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
STAGE1_EPOCHS = 5

for epoch in range(STAGE1_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer_s1)
    val_loss, val_acc = evaluate(model, val_loader)
    print(f"[S1 {epoch+1}/{STAGE1_EPOCHS}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

## 9. Stage 2 — Fine-Tune Entire Model

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer_s2 = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if "head" not in n], "lr": 1e-5},
    {"params": model.head.parameters(), "lr": 1e-4},
], weight_decay=1e-2)

STAGE2_EPOCHS = 15
scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s2, T_max=STAGE2_EPOCHS)

best_val_loss = float("inf")
patience, patience_counter = 5, 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(STAGE2_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer_s2, scheduler_s2)
    val_loss, val_acc = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"[S2 {epoch+1}/{STAGE2_EPOCHS}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
        print("  → saved best model")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  → early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_model.pt", weights_only=True))
print("Loaded best checkpoint.")

## 10. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"], label="Val Loss")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True)

ax2.plot(history["train_acc"], label="Train Acc")
ax2.plot(history["val_acc"], label="Val Acc")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 11. Test Set Evaluation

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

In [ ]:
class_names = full_dataset.classes
report = classification_report(all_labels, all_preds, target_names=class_names)
print(report)

with open("classification_report.txt", "w") as f:
    f.write(report)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"Sensitivity (TPR): {sensitivity:.4f}")
print(f"Specificity (TNR): {specificity:.4f}")

In [ ]:
roc_auc = roc_auc_score(all_labels, all_probs)
fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.4f}", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.show()

print(f"ROC-AUC: {roc_auc:.4f}")

## 12. Export Model

In [ ]:
MODEL_VERSION = "v0.1.0"
export_path = f"model_{MODEL_VERSION}.pt"
torch.save(model.state_dict(), export_path)
print(f"Saved: {export_path} ({os.path.getsize(export_path) / 1e6:.1f} MB)")

## 13. Download Artifacts

In [ ]:
from google.colab import files

for f in [export_path, "classification_report.txt", "confusion_matrix.png", "roc_curve.png", "training_curves.png"]:
    files.download(f)
    print(f"Downloaded: {f}")